## Presentación
### TC5035 Proyecto Integrador
- Dra. Grettel Barceló Alonso
- Dr. Luis Eduardo Falcón Morales

### Asesora
- Dra. María de la Paz Rico Fernández

### Equipo 31
- A00194173 Adriana González Ugalde

- A01123424 José Alberto Herrera Bernal

- A00534649 Carlos Alberto Parra Arredondo

# Curado de Imágenes: Limpieza, mejora y calidad visual del dataset con OpenCV

Este notebook realiza una auditoría visual profesional del dataset antes del entrenamiento.

El objetivo no es solo mejorar imágenes, sino demostrar que el dataset fue revisado,
medido, categorizado y preparado de forma reproducible.

El proceso incluye:

1. Validación de estructura del dataset.
2. Lectura segura de imágenes en RGB.
3. Análisis de calidad visual.
4. Detección de imágenes problemáticas.
5. Clasificación de calidad por imagen.
6. Análisis por carpeta / clase.
7. Mejora adaptativa con OpenCV.
8. Guardado de imágenes procesadas.
9. Validación final de imágenes generadas.
10. Exportación de metadata para documentación.

In [ ]:
!pip install opencv-python pillow tqdm pandas matplotlib numpy

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageOps
import json
import os

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

In [ ]:
# Montar Google  Drive
from google.colab import drive
from pathlib import Path

if Path("/content/drive/MyDrive").exists():
    print("Google Drive ya está montado.")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")

print("DRIVE_ROOT:", DRIVE_ROOT)

In [ ]:
# Rutas del proyecto
PROJECT_PATH = DRIVE_ROOT / "proyecto_integrador"

DATASET_PATH = PROJECT_PATH / "images"

# Salida única: versión mejorada V2
OUTPUT_PATH_V2 = PROJECT_PATH / "images_curated_rgb_v2"

REPORTS_PATH = PROJECT_PATH / "opencv_curation_reports"

METADATA_RAW_PATH = REPORTS_PATH / "metadata_raw_quality.csv"

METADATA_CURATED_V2_PATH = REPORTS_PATH / "metadata_curated_quality_v2.csv"

SUMMARY_V2_PATH = REPORTS_PATH / "dataset_curation_summary_v2.csv"

CONFIG_V2_PATH = REPORTS_PATH / "opencv_curation_config_v2.json"

EXECUTIVE_SUMMARY_V2_PATH = REPORTS_PATH / "executive_curation_summary_v2.csv"

print("PROJECT_PATH:", PROJECT_PATH)
print("DATASET_PATH:", DATASET_PATH)
print("OUTPUT_PATH_V2:", OUTPUT_PATH_V2)
print("REPORTS_PATH:", REPORTS_PATH)
print("Existe dataset:", DATASET_PATH.exists())


In [ ]:
# Valicación de ruta
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"No existe DATASET_PATH: {DATASET_PATH}. "
        "Revisa la ruta exacta del dataset antes de continuar."
    )

REPORTS_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH_V2.mkdir(parents=True, exist_ok=True)

In [ ]:
# Configuración del pipeline V2
# V2 combina CLAHE, contraste/brillo moderado, saturación ligera, nitidez y resize con padding.

CURATION_CONFIG_V2 = {
    "target_size": 224,
    "padding_value": 114,
    "jpeg_quality": 95,

    "image_extensions": [".jpg", ".jpeg", ".png"],

    # Umbrales de calidad visual para auditoría y clasificación
    "dark_brightness_threshold": 60,
    "bright_brightness_threshold": 190,
    "low_contrast_threshold": 35,
    "low_sharpness_threshold": 80,
    "low_colorfulness_threshold": 18,

    # Umbrales para clasificación de calidad
    "excellent_sharpness_threshold": 500,
    "good_sharpness_threshold": 150,
    "excellent_contrast_threshold": 55,
    "good_contrast_threshold": 40,
    "min_width": 100,
    "min_height": 100,

    # V2: Equalización adaptativa de histograma
    "apply_clahe": True,
    "clahe_clip_limit": 2.5,
    "clahe_tile_grid_size": (8, 8),

    # V2: Contraste y brillo moderado
    "apply_contrast": True,
    "contrast_alpha": 1.08,
    "brightness_beta": 4,

    # V2: Color moderado
    "apply_saturation": True,
    "saturation_factor": 1.05,

    # V2: Nitidez
    "apply_sharpening": True,
    "sharpen_amount": 1.35,
    "sharpen_blur_sigma": 1.0,
}


In [ ]:
# Guardamos la configuración V2
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

with open(CONFIG_V2_PATH, "w") as f:
    json.dump(CURATION_CONFIG_V2, f, indent=4)

print("Configuración V2 guardada en:", CONFIG_V2_PATH)


## Convención de color

Este notebook trabaja internamente en RGB.

OpenCV normalmente lee y guarda imágenes en BGR. Para evitar errores de canales:

- Las imágenes se cargan con PIL y se convierten a RGB.
- Las métricas se calculan sobre RGB.
- Las mejoras se aplican sobre RGB.
- Antes de guardar con `cv2.imwrite`, se convierte RGB → BGR.
- Para validar imágenes guardadas, se leen con OpenCV y se convierten BGR → RGB.

Esto evita que las imágenes finales queden con colores invertidos.

In [ ]:
# Buscamos las imágenes en el dataset
image_paths = []

for ext in CURATION_CONFIG_V2["image_extensions"]:
    image_paths.extend(DATASET_PATH.rglob(f"*{ext}"))

image_paths = sorted(image_paths)

print("Total de imágenes encontradas:", len(image_paths))

if len(image_paths) == 0:
    raise ValueError("No se encontraron imágenes. Revisa extensiones o ruta del dataset.")

print("Primera imagen:", image_paths[0])

In [ ]:
# Resumen de las carpetas
class_folders = sorted([p for p in DATASET_PATH.iterdir() if p.is_dir()])

print("Número de carpetas/clases:", len(class_folders))

for folder in class_folders[:10]:
    print(folder.name)

In [ ]:
# Conteo de las carpetas
class_counts = []

for folder in class_folders:
    count = 0
    for ext in CURATION_CONFIG_V2["image_extensions"]:
        count += len(list(folder.glob(f"*{ext}")))

    class_counts.append({
        "folder": folder.name,
        "image_count": count
    })

class_counts_df = pd.DataFrame(class_counts).sort_values(
    "image_count",
    ascending=False
)

class_counts_df.head()

In [ ]:
# Graficamos distribución de imágenes por carpeta/clase

plt.figure(figsize=(12, 5))
plt.hist(class_counts_df["image_count"], bins=30)
plt.title("Distribución de imágenes por carpeta/clase")
plt.xlabel("Número de imágenes")
plt.ylabel("Frecuencia")
plt.grid(True)
plt.show()

In [ ]:
# Función de carga de imágnees
def load_image_as_rgb(image_path):
    """
    Carga una imagen respetando orientación EXIF.
    Devuelve imagen en RGB.
    """
    try:
        image = Image.open(image_path)
        image = ImageOps.exif_transpose(image)
        image = image.convert("RGB")
        return np.array(image)

    except Exception as e:
        print(f"Error leyendo imagen {image_path}: {e}")
        return None

In [ ]:
# Función de guardado de imágenes BGR
def save_rgb_image_with_opencv(image_rgb, output_path, jpeg_quality=95):
    """
    Guarda una imagen RGB usando OpenCV.
    Convierte RGB -> BGR antes de guardar.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    success = cv2.imwrite(
        str(output_path),
        image_bgr,
        [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality]
    )

    if not success:
        raise ValueError(f"No se pudo guardar la imagen en: {output_path}")

In [ ]:
# Función de carga de imágenes
def load_saved_image_as_rgb(image_path):
    """
    Lee una imagen guardada con OpenCV y la devuelve en RGB.
    """
    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        raise ValueError(f"No se pudo leer la imagen: {image_path}")

    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


In [ ]:
# Función para calcular el brillo
def calculate_brightness(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    return float(np.mean(gray))

In [ ]:
# Función para calcular el contraste
def calculate_contrast(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    return float(np.std(gray))

In [ ]:
# Función para calcular sharpness
def calculate_sharpness(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    return float(laplacian.var())

In [ ]:
# Función para calcular colorfulness
def calculate_colorfulness(image_rgb):
    image = image_rgb.astype("float")

    R = image[:, :, 0]
    G = image[:, :, 1]
    B = image[:, :, 2]

    rg = np.abs(R - G)
    yb = np.abs(0.5 * (R + G) - B)

    std_rg = np.std(rg)
    std_yb = np.std(yb)

    mean_rg = np.mean(rg)
    mean_yb = np.mean(yb)

    return float(
        np.sqrt(std_rg**2 + std_yb**2)
        + 0.3 * np.sqrt(mean_rg**2 + mean_yb**2)
    )

In [ ]:
# Función para calcular edg density
def calculate_edge_density(image_rgb):
    """
    Mide densidad de bordes usando Canny.
    Puede indicar textura, complejidad o ruido visual.
    """
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)

    edge_pixels = np.sum(edges > 0)
    total_pixels = edges.shape[0] * edges.shape[1]

    return float(edge_pixels / total_pixels)

In [ ]:
# Función para obtener métricas de calidad
def get_image_quality_metrics(image_rgb):
    height, width = image_rgb.shape[:2]

    return {
        "width": width,
        "height": height,
        "aspect_ratio": width / height if height != 0 else None,
        "brightness": calculate_brightness(image_rgb),
        "contrast": calculate_contrast(image_rgb),
        "sharpness": calculate_sharpness(image_rgb),
        "colorfulness": calculate_colorfulness(image_rgb),
        "edge_density": calculate_edge_density(image_rgb)
    }

In [ ]:
# Función para clasificar la calidad de las imágenes
def classify_image_quality(metrics, config):
    """
    Clasifica la calidad visual de una imagen.
    """

    issues = []

    if metrics["width"] < config["min_width"] or metrics["height"] < config["min_height"]:
        issues.append("very_small")

    if metrics["brightness"] < config["dark_brightness_threshold"]:
        issues.append("dark")

    if metrics["brightness"] > config["bright_brightness_threshold"]:
        issues.append("too_bright")

    if metrics["contrast"] < config["low_contrast_threshold"]:
        issues.append("low_contrast")

    if metrics["sharpness"] < config["low_sharpness_threshold"]:
        issues.append("blurry")

    if metrics["colorfulness"] < config["low_colorfulness_threshold"]:
        issues.append("low_color")

    if len(issues) == 0:
        if (
            metrics["sharpness"] >= config["excellent_sharpness_threshold"]
            and metrics["contrast"] >= config["excellent_contrast_threshold"]
        ):
            quality_label = "excellent"
        elif (
            metrics["sharpness"] >= config["good_sharpness_threshold"]
            and metrics["contrast"] >= config["good_contrast_threshold"]
        ):
            quality_label = "good"
        else:
            quality_label = "acceptable"
    else:
        quality_label = "needs_review"

    return quality_label, issues

In [ ]:
# Auditoría del dataset
raw_records = []

for image_path in tqdm(image_paths):
    image_rgb = load_image_as_rgb(image_path)

    if image_rgb is None:
        raw_records.append({
            "image_path": str(image_path),
            "folder": image_path.parent.name,
            "filename": image_path.name,
            "status": "error_loading"
        })
        continue

    metrics = get_image_quality_metrics(image_rgb)
    quality_label, issues = classify_image_quality(metrics, CURATION_CONFIG_V2)

    record = {
        "image_path": str(image_path),
        "folder": image_path.parent.name,
        "filename": image_path.name,
        "status": "loaded",
        "quality_label": quality_label,
        "issues": ",".join(issues)
    }

    record.update(metrics)

    raw_records.append(record)

raw_quality_df = pd.DataFrame(raw_records)

raw_quality_df.to_csv(METADATA_RAW_PATH, index=False)

raw_quality_df.head()

In [ ]:
# Resumen de la calidad original
raw_quality_df["status"].value_counts()


In [ ]:

raw_quality_df["quality_label"].value_counts()

In [ ]:
issue_counts = (
    raw_quality_df["issues"]
    .fillna("")
    .str.get_dummies(sep=",")
    .sum()
    .sort_values(ascending=False)
)

issue_counts

In [ ]:
# Función para graficar métricas
def plot_metric_distribution(df, column, title):
    plt.figure(figsize=(10, 5))
    plt.hist(df[column].dropna(), bins=50)
    plt.title(title)
    plt.xlabel(column)
    plt.ylabel("Frecuencia")
    plt.grid(True)
    plt.show()

In [ ]:
valid_raw_df = raw_quality_df[raw_quality_df["status"] == "loaded"].copy()

plot_metric_distribution(valid_raw_df, "brightness", "Distribución de brillo")
plot_metric_distribution(valid_raw_df, "contrast", "Distribución de contraste")
plot_metric_distribution(valid_raw_df, "sharpness", "Distribución de nitidez")
plot_metric_distribution(valid_raw_df, "colorfulness", "Distribución de colorfulness")
plot_metric_distribution(valid_raw_df, "edge_density", "Distribución de densidad de bordes")

In [ ]:
# Análisis por clase/carpetas
quality_by_folder = (
    valid_raw_df
    .groupby("folder")
    .agg(
        image_count=("image_path", "count"),
        avg_brightness=("brightness", "mean"),
        avg_contrast=("contrast", "mean"),
        avg_sharpness=("sharpness", "mean"),
        avg_colorfulness=("colorfulness", "mean"),
        avg_edge_density=("edge_density", "mean")
    )
    .reset_index()
)

quality_by_folder.head()

In [ ]:
quality_by_folder.sort_values("avg_sharpness").head(10)

In [ ]:
quality_by_folder.sort_values("avg_contrast").head(10)

In [ ]:
needs_review_by_folder = (
    valid_raw_df[valid_raw_df["quality_label"] == "needs_review"]
    .groupby("folder")
    .size()
    .reset_index(name="needs_review_count")
)

quality_by_folder = quality_by_folder.merge(
    needs_review_by_folder,
    on="folder",
    how="left"
)

quality_by_folder["needs_review_count"] = quality_by_folder["needs_review_count"].fillna(0)

quality_by_folder["needs_review_ratio"] = (
    quality_by_folder["needs_review_count"] / quality_by_folder["image_count"]
)

quality_by_folder.sort_values("needs_review_ratio", ascending=False).head(10)

In [ ]:
# Función para validad muestras
def show_image_grid_from_df(df, path_column="image_path", n=12, title_prefix="Imagen"):
    if len(df) == 0:
        print("No hay imágenes para mostrar.")
        return

    sample_df = df.sample(min(n, len(df)), random_state=42)

    cols = 3
    rows = int(np.ceil(len(sample_df) / cols))

    plt.figure(figsize=(12, 4 * rows))

    for i, (_, row) in enumerate(sample_df.iterrows()):
        image_rgb = load_image_as_rgb(row[path_column])

        plt.subplot(rows, cols, i + 1)
        plt.imshow(image_rgb)
        plt.title(f"{title_prefix}: {row['folder']}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Imágenes oscuras
dark_df = valid_raw_df[valid_raw_df["issues"].str.contains("dark", na=False)]
show_image_grid_from_df(dark_df, n=9, title_prefix="Oscura")

In [ ]:
# Imágenes borrosas
blurry_df = valid_raw_df[valid_raw_df["issues"].str.contains("blurry", na=False)]
show_image_grid_from_df(blurry_df, n=9, title_prefix="Borrosa")

In [ ]:
# Imágenes bajo contraste
low_contrast_df = valid_raw_df[valid_raw_df["issues"].str.contains("low_contrast", na=False)]
show_image_grid_from_df(low_contrast_df, n=9, title_prefix="Bajo contraste")

In [ ]:
# Imágenes bajo color
low_color_df = valid_raw_df[valid_raw_df["issues"].str.contains("low_color", na=False)]
show_image_grid_from_df(low_color_df, n=9, title_prefix="Bajo color")

In [ ]:
def resize_with_padding(image_rgb, target_size=224, padding_value=114):
    h, w = image_rgb.shape[:2]

    scale = target_size / max(h, w)

    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(
        image_rgb,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )

    padded = np.full(
        (target_size, target_size, 3),
        padding_value,
        dtype=np.uint8
    )

    top = (target_size - new_h) // 2
    left = (target_size - new_w) // 2

    padded[top:top + new_h, left:left + new_w] = resized

    return padded

In [ ]:
def apply_clahe_lab_rgb(image_rgb, clip_limit=2.0, tile_grid_size=(8, 8)):
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)

    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=tile_grid_size
    )

    l_clahe = clahe.apply(l_channel)

    lab_clahe = cv2.merge((l_clahe, a_channel, b_channel))

    return cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)

In [ ]:
def adjust_contrast_brightness_rgb(image_rgb, alpha=1.10, beta=3):
    return cv2.convertScaleAbs(image_rgb, alpha=alpha, beta=beta)

In [ ]:
def adjust_saturation_rgb(image_rgb, saturation_factor=1.08):
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)

    hsv[:, :, 1] *= saturation_factor
    hsv[:, :, 1] = np.clip(hsv[:, :, 1], 0, 255)

    hsv = hsv.astype(np.uint8)

    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

In [ ]:
def apply_denoising_rgb(image_rgb):
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    denoised_bgr = cv2.fastNlMeansDenoisingColored(
        image_bgr,
        None,
        h=5,
        hColor=5,
        templateWindowSize=7,
        searchWindowSize=21
    )

    return cv2.cvtColor(denoised_bgr, cv2.COLOR_BGR2RGB)

In [ ]:
def apply_unsharp_mask_rgb(image_rgb, amount=1.3, sigma=1.0):
    blurred = cv2.GaussianBlur(image_rgb, (0, 0), sigma)

    sharpened = cv2.addWeighted(
        image_rgb,
        amount,
        blurred,
        1 - amount,
        0
    )

    return np.clip(sharpened, 0, 255).astype(np.uint8)

In [ ]:
# Pipeline V2: CLAHE + Unsharp Mask
def preprocess_image_v2_rgb(image_rgb, config):
    """
    Pipeline V2 de mejora visual.

    Técnicas principales:
    1. CLAHE en canal L del espacio LAB.
    2. Ajuste moderado de contraste y brillo.
    3. Saturación ligera.
    4. Nitidez con Unsharp Mask.
    5. Resize con padding gris.

    Entrada:
        image_rgb: imagen en RGB.

    Salida:
        processed_rgb: imagen procesada en RGB.
    """
    processed_rgb = image_rgb.copy()

    if config["apply_clahe"]:
        processed_rgb = apply_clahe_lab_rgb(
            processed_rgb,
            clip_limit=config["clahe_clip_limit"],
            tile_grid_size=config["clahe_tile_grid_size"]
        )

    if config["apply_contrast"]:
        processed_rgb = adjust_contrast_brightness_rgb(
            processed_rgb,
            alpha=config["contrast_alpha"],
            beta=config["brightness_beta"]
        )

    if config["apply_saturation"]:
        processed_rgb = adjust_saturation_rgb(
            processed_rgb,
            saturation_factor=config["saturation_factor"]
        )

    if config["apply_sharpening"]:
        processed_rgb = apply_unsharp_mask_rgb(
            processed_rgb,
            amount=config["sharpen_amount"],
            sigma=config["sharpen_blur_sigma"]
        )

    processed_rgb = resize_with_padding(
        processed_rgb,
        target_size=config["target_size"],
        padding_value=config["padding_value"]
    )

    return processed_rgb

In [ ]:
# Prueba visual del pipeline V2
sample_image_path = image_paths[0]
sample_image_rgb = load_image_as_rgb(sample_image_path)

processed_v2_rgb = preprocess_image_v2_rgb(
    sample_image_rgb,
    CURATION_CONFIG_V2
)

before_metrics_v2 = get_image_quality_metrics(sample_image_rgb)
after_metrics_v2 = get_image_quality_metrics(processed_v2_rgb)

after_quality_label_v2, after_issues_v2 = classify_image_quality(
    after_metrics_v2,
    CURATION_CONFIG_V2
)

comparison_v2_df = pd.DataFrame({
    "metric": before_metrics_v2.keys(),
    "before": before_metrics_v2.values(),
    "after_v2": after_metrics_v2.values()
})

print("Calidad después V2:", after_quality_label_v2)
print("Issues después V2:", after_issues_v2)

comparison_v2_df


In [ ]:
# Visualización de ejemplo: Original vs V2
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(sample_image_rgb)
plt.title("Original RGB")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(processed_v2_rgb)
plt.title("Curada V2: CLAHE + nitidez")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
OUTPUT_PATH_V2.mkdir(parents=True, exist_ok=True)

# Procesamiento de todo el dataset con V2
curated_v2_records = []

for image_path in tqdm(image_paths):
    try:
        relative_path = image_path.relative_to(DATASET_PATH)
        output_image_path = OUTPUT_PATH_V2 / relative_path

        image_rgb = load_image_as_rgb(image_path)

        if image_rgb is None:
            curated_v2_records.append({
                "image_path": str(image_path),
                "output_path": str(output_image_path),
                "folder": image_path.parent.name,
                "filename": image_path.name,
                "status": "error_loading"
            })
            continue

        before_metrics = get_image_quality_metrics(image_rgb)

        processed_rgb = preprocess_image_v2_rgb(
            image_rgb,
            CURATION_CONFIG_V2
        )

        after_metrics = get_image_quality_metrics(processed_rgb)

        after_quality_label, after_issues = classify_image_quality(
            after_metrics,
            CURATION_CONFIG_V2
        )

        save_rgb_image_with_opencv(
            processed_rgb,
            output_image_path,
            jpeg_quality=CURATION_CONFIG_V2["jpeg_quality"]
        )

        record = {
            "image_path": str(image_path),
            "output_path": str(output_image_path),
            "folder": image_path.parent.name,
            "filename": image_path.name,
            "status": "processed",

            "pipeline_version": "v2_clahe_unsharp_mask",
            "internal_color_space": "RGB",
            "save_conversion": "RGB_to_BGR_before_cv2_imwrite",

            "quality_label_after": after_quality_label,
            "issues_after": ",".join(after_issues),

            "applied_clahe": CURATION_CONFIG_V2["apply_clahe"],
            "applied_contrast": CURATION_CONFIG_V2["apply_contrast"],
            "applied_saturation": CURATION_CONFIG_V2["apply_saturation"],
            "applied_sharpening": CURATION_CONFIG_V2["apply_sharpening"],

            "clahe_clip_limit": CURATION_CONFIG_V2["clahe_clip_limit"],
            "contrast_alpha_used": CURATION_CONFIG_V2["contrast_alpha"],
            "brightness_beta_used": CURATION_CONFIG_V2["brightness_beta"],
            "saturation_factor_used": CURATION_CONFIG_V2["saturation_factor"],
            "sharpen_amount_used": CURATION_CONFIG_V2["sharpen_amount"],
        }

        for key, value in before_metrics.items():
            record[f"{key}_before"] = value

        for key, value in after_metrics.items():
            record[f"{key}_after"] = value

        curated_v2_records.append(record)

    except Exception as e:
        curated_v2_records.append({
            "image_path": str(image_path),
            "folder": image_path.parent.name,
            "filename": image_path.name,
            "status": "error_processing",
            "error": str(e)
        })

curated_v2_df = pd.DataFrame(curated_v2_records)

curated_v2_df.to_csv(METADATA_CURATED_V2_PATH, index=False)

curated_v2_df.head()

In [ ]:
# V2
valid_v2_df = curated_v2_df[curated_v2_df["status"] == "processed"].copy()

print("Total procesadas correctamente V2:", len(valid_v2_df))
print("Errores V2:", len(curated_v2_df) - len(valid_v2_df))

In [ ]:
summary_v2 = pd.DataFrame({
    "metric": [
        "brightness",
        "contrast",
        "sharpness",
        "colorfulness",
        "edge_density"
    ],
    "before_mean": [
        valid_v2_df["brightness_before"].mean(),
        valid_v2_df["contrast_before"].mean(),
        valid_v2_df["sharpness_before"].mean(),
        valid_v2_df["colorfulness_before"].mean(),
        valid_v2_df["edge_density_before"].mean()
    ],
    "after_v2_mean": [
        valid_v2_df["brightness_after"].mean(),
        valid_v2_df["contrast_after"].mean(),
        valid_v2_df["sharpness_after"].mean(),
        valid_v2_df["colorfulness_after"].mean(),
        valid_v2_df["edge_density_after"].mean()
    ]
})

summary_v2["difference_v2"] = (
    summary_v2["after_v2_mean"] - summary_v2["before_mean"]
)

summary_v2

In [ ]:
summary_v2.to_csv(SUMMARY_V2_PATH, index=False)

print("Resumen V2 guardado en:", SUMMARY_V2_PATH)

In [ ]:
# Acciones aplicadas por el pipeline V2
action_columns_v2 = [
    "applied_clahe",
    "applied_contrast",
    "applied_saturation",
    "applied_sharpening"
]

valid_v2_df[action_columns_v2].sum().sort_values(ascending=False)


In [ ]:
# Calidad después de las imágenes curadas
curated_v2_df["quality_label_after"].value_counts()

In [ ]:
# Datos adicionales de calidad después de V2
issues_after_counts_v2 = (
    valid_v2_df["issues_after"]
    .fillna("")
    .str.get_dummies(sep=",")
    .sum()
    .sort_values(ascending=False)
)

issues_after_counts_v2


In [ ]:
# Procesamiento de imágenes V2: validación de archivos generados
processed_image_paths_v2 = []

for ext in CURATION_CONFIG_V2["image_extensions"]:
    processed_image_paths_v2.extend(OUTPUT_PATH_V2.rglob(f"*{ext}"))

processed_image_paths_v2 = sorted(processed_image_paths_v2)

print("Imágenes procesadas V2 encontradas:", len(processed_image_paths_v2))


In [ ]:
# Muestra aleatoria para revisar imágenes V2 procesadas
if len(processed_image_paths_v2) > 0:
    sample_processed_v2 = np.random.choice(
        processed_image_paths_v2,
        size=min(12, len(processed_image_paths_v2)),
        replace=False
    )

    plt.figure(figsize=(12, 16))

    for i, img_path in enumerate(sample_processed_v2):
        image_rgb = load_saved_image_as_rgb(img_path)

        plt.subplot(4, 3, i + 1)
        plt.imshow(image_rgb)
        plt.title(Path(img_path).parent.name)
        plt.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron imágenes procesadas V2 para mostrar.")


In [ ]:
# Antes y después usando una muestra del pipeline V2
sample_rows_v2 = valid_v2_df.sample(
    min(6, len(valid_v2_df)),
    random_state=42
)

plt.figure(figsize=(12, 18))

for i, (_, row) in enumerate(sample_rows_v2.iterrows()):
    original_rgb = load_image_as_rgb(row["image_path"])
    processed_rgb = load_saved_image_as_rgb(row["output_path"])

    plt.subplot(len(sample_rows_v2), 2, i * 2 + 1)
    plt.imshow(original_rgb)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(len(sample_rows_v2), 2, i * 2 + 2)
    plt.imshow(processed_rgb)
    plt.title("Curada V2")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Validamos tamaños finales V2
shape_records_v2 = []

for img_path in tqdm(processed_image_paths_v2[:1000]):
    image_rgb = load_saved_image_as_rgb(img_path)
    shape_records_v2.append(image_rgb.shape)

unique_shapes_v2 = set(shape_records_v2)

print("Shapes únicos encontrados en muestra V2:")
print(unique_shapes_v2)


In [ ]:
# Validamos archivos corruptos V2
corrupt_files_v2 = []

for img_path in tqdm(processed_image_paths_v2):
    try:
        image_rgb = load_saved_image_as_rgb(img_path)

        if image_rgb is None:
            corrupt_files_v2.append(str(img_path))

    except Exception:
        corrupt_files_v2.append(str(img_path))

print("Archivos corruptos V2 encontrados:", len(corrupt_files_v2))

corrupt_files_v2[:10]


In [ ]:
# Generación del resumen ejecutivo V2
executive_summary_v2 = {
    "pipeline_version": "v2_clahe_unsharp_mask",
    "total_original_images": len(image_paths),
    "total_processed_images_v2": len(processed_image_paths_v2),
    "total_classes_or_folders": len(class_folders),
    "processing_success_count_v2": int((curated_v2_df["status"] == "processed").sum()),
    "processing_error_count_v2": int((curated_v2_df["status"] != "processed").sum()),
    "avg_brightness_before": valid_v2_df["brightness_before"].mean(),
    "avg_brightness_after_v2": valid_v2_df["brightness_after"].mean(),
    "avg_contrast_before": valid_v2_df["contrast_before"].mean(),
    "avg_contrast_after_v2": valid_v2_df["contrast_after"].mean(),
    "avg_sharpness_before": valid_v2_df["sharpness_before"].mean(),
    "avg_sharpness_after_v2": valid_v2_df["sharpness_after"].mean(),
    "avg_colorfulness_before": valid_v2_df["colorfulness_before"].mean(),
    "avg_colorfulness_after_v2": valid_v2_df["colorfulness_after"].mean(),
}

executive_summary_v2_df = pd.DataFrame([executive_summary_v2])

executive_summary_v2_df


In [ ]:
# Guardamos resumen ejecutivo V2
executive_summary_v2_df.to_csv(EXECUTIVE_SUMMARY_V2_PATH, index=False)

print("Resumen ejecutivo V2 guardado en:", EXECUTIVE_SUMMARY_V2_PATH)


In [ ]:
# Resultados finales V2
print("Proceso de curado visual V2 terminado.")
print("")
print("Dataset original:")
print(DATASET_PATH)
print("")
print("Dataset curado V2:")
print(OUTPUT_PATH_V2)
print("")
print("Reportes generados:")
print(REPORTS_PATH)
print("")
print("Archivos principales V2:")
print("- Metadata original:", METADATA_RAW_PATH)
print("- Metadata curada V2:", METADATA_CURATED_V2_PATH)
print("- Resumen comparativo V2:", SUMMARY_V2_PATH)
print("- Configuración V2:", CONFIG_V2_PATH)
print("- Resumen ejecutivo V2:", EXECUTIVE_SUMMARY_V2_PATH)
print("")
print("Convención de color:")
print("- Procesamiento interno en RGB.")
print("- Guardado con conversión RGB -> BGR para OpenCV.")
print("- Validación posterior leyendo BGR -> RGB.")


Se realizó una curaduría visual del dataset usando OpenCV sobre 20,580 imágenes distribuidas en 120 clases. El proceso genera una única salida curada con el pipeline V2, estandarizada a 224x224 píxeles con padding gris, calidad JPEG 95 y control explícito de color RGB.

La auditoría inicial permite identificar imágenes con baja iluminación, bajo contraste, baja nitidez o baja riqueza de color. Estos indicadores se conservan como métricas de trazabilidad para justificar el preprocesamiento aplicado.

El pipeline V2 aplica CLAHE en el canal de luminancia, ajuste moderado de contraste y brillo, saturación ligera y nitidez mediante Unsharp Mask. Después, todas las imágenes se normalizan a un tamaño fijo de 224x224 píxeles.

Esta versión queda como la versión final del dataset curado. La versión anterior fue eliminada del flujo para evitar duplicidad, confusión en los reportes y comparaciones innecesarias. A partir de este notebook, los archivos principales son `images_curated_rgb_v2`, `metadata_curated_quality_v2.csv`, `dataset_curation_summary_v2.csv`, `opencv_curation_config_v2.json` y `executive_curation_summary_v2.csv`.


# Indicadores finales de curaduría visual — Dataset V2

La versión final del pipeline de curaduría visual corresponde a la **Versión 2**, la cual aplica técnicas de mejora de imagen con OpenCV, incluyendo CLAHE, ajuste de contraste, saturación, nitidez y estandarización del tamaño de entrada para modelos de visión computacional.

## Indicadores generales del procesamiento

| Indicador | Resultado |
|---|---:|
| Imágenes originales | 20,580 |
| Imágenes procesadas en V2 | 20,580 |
| Clases / razas | 120 |
| Errores de procesamiento | 0 |
| Archivos corruptos después del procesamiento | 0 |
| Tamaño final de imagen | 224 × 224 × 3 |
| Pipeline final | CLAHE + contraste + saturación + sharpening |

## Comparación de calidad visual: original vs V2

| Métrica | Dataset original | Dataset curado V2 | Cambio |
|---|---:|---:|---:|
| Brillo promedio | 115.30 | 128.88 | +13.58 |
| Contraste promedio | 57.64 | 54.72 | -2.91 |
| Nitidez promedio | 2455.39 | 4375.40 | +1920.01 |
| Colorfulness promedio | 33.04 | 34.91 | +1.87 |
| Densidad de bordes | 0.123 | 0.188 | +0.065 |

## Distribución de calidad después de V2

| Categoría de calidad | Cantidad de imágenes |
|---|---:|
| Excellent | 8,168 |
| Good | 9,264 |
| Acceptable | 413 |
| Needs review | 2,735 |

## Reducción de problemas visuales

| Problema detectado | Antes | Después V2 |
|---|---:|---:|
| Bajo contraste | 843 | 134 |
| Imágenes oscuras | 451 | 2 |
| Imágenes demasiado brillantes | 143 | 49 |
| Bajo color | 2,920 | 2,599 |

## Conclusión del Paso 01

Los resultados muestran que la **Versión 2 del pipeline de curaduría visual es estable, efectiva y adecuada como versión final del dataset**. Se procesaron correctamente las **20,580 imágenes**, distribuidas en **120 clases**, sin errores de procesamiento ni archivos corruptos posteriores. Esto confirma que el flujo es técnicamente confiable y reproducible.

La mejora más significativa se observa en la **nitidez**, que aumentó de **2455.39 a 4375.40**, y en la **densidad de bordes**, que pasó de **0.123 a 0.188**. Estos cambios indican que las imágenes resultantes presentan mayor definición, mejor estructura visual y rasgos más claros, aspectos relevantes para modelos de visión computacional enfocados en clasificación de razas de perros.

Además, la V2 redujo considerablemente problemas críticos de calidad visual. Las imágenes oscuras disminuyeron de **451 a solo 2**, mientras que los casos de bajo contraste bajaron de **843 a 134**. Esto demuestra que el pipeline mejora la consistencia visual del dataset y reduce condiciones que podrían afectar negativamente el entrenamiento del modelo.

Aunque el contraste global promedio disminuyó ligeramente, esto no representa necesariamente una pérdida de calidad, ya que el uso de **CLAHE** mejora el contraste local y permite equilibrar mejor la iluminación en distintas regiones de la imagen. Por esta razón, la evaluación debe considerar no solo el contraste global, sino también la nitidez, la densidad de bordes, la reducción de errores visuales y la estabilidad del procesamiento.

En conclusión, la **Versión 2** se selecciona como la versión final del dataset curado porque ofrece una mejora clara en definición visual, reduce problemas críticos de iluminación y contraste, mantiene la integridad de todas las imágenes y deja el dataset preparado para la siguiente fase: entrenamiento y evaluación de modelos de clasificación de razas.